# Inspect Dense LeJEPA checkpoints (CPU inference)

Use this after `examples/dense_lejepa_ddp_spawn.py` (or any run that produced `checkpoint_*.pt`).

- Pick a **`checkpoint_latest.pt`** (or a specific `checkpoint_epoch_####.pt`).
- Load weights on **CPU** and run a short forward on synthetic disc-square images (same as training) or your own grayscale files.
- Visualize **per-hook** dense latents (L2 norm across channels, view 0).

**Tip:** Training often uses `latent.step_mode="rotate"`. Here we temporarily set **`joint`** so one forward fills every key in `latents_by_source`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Repo root (parent of examples/ if you launch from examples/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from examples.dense_lejepa_ddp_spawn import load_dense_lejepa_from_checkpoint
from local_conv_attention import DiscSquareDataset

device = torch.device("cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("device:", device)

In [ ]:
# --- user settings ---
# Default spawn layout: PROJECT_ROOT / "examples" / "dense_lejepa_ddp_outputs" / <timestamp> /
# If you trained with ``--output-dir ./my_run``, set OUTPUT_ROOT = PROJECT_ROOT / "my_run"
OUTPUT_ROOT = PROJECT_ROOT / "examples" / "dense_lejepa_ddp_outputs"
# File path, or None to auto-pick newest checkpoint_latest.pt under OUTPUT_ROOT
# Relative paths resolve against cwd first, then PROJECT_ROOT (see next cell).
CHECKPOINT_PATH: Path | None = None  # e.g. Path("my_run/checkpoint_latest.pt")

NUM_IMAGES = 4  # synthetic samples
IMAGE_SIZE = 128  # training default for spawn demo; change if your checkpoint used another size

# Optional: grayscale image paths (PNG/JPG). Resized to IMAGE_SIZE x IMAGE_SIZE
CUSTOM_IMAGE_PATHS: list[Path] = []

In [ ]:
def resolve_maybe_relative(path: Path | str) -> Path:
    """Notebook CWD is often not the repo root; try PROJECT_ROOT for relative paths."""
    p = Path(path).expanduser()
    if p.is_file():
        return p.resolve()
    if not p.is_absolute():
        cand = (PROJECT_ROOT / p).resolve()
        if cand.is_file():
            return cand
    return p.resolve()


def find_default_checkpoint(root: Path) -> Path | None:
    """Handles both layouts: (1) ``output_dir/checkpoint_latest.pt`` from ``--output-dir ./my_run``,
    (2) ``.../dense_lejepa_ddp_outputs/<stamp>/checkpoint_latest.pt`` default spawn layout.
    """
    if not root.is_dir():
        return None
    candidates: list[Path] = []
    direct = root / "checkpoint_latest.pt"
    if direct.is_file():
        candidates.append(direct)
    for run in root.iterdir():
        if not run.is_dir():
            continue
        nested = run / "checkpoint_latest.pt"
        if nested.is_file():
            candidates.append(nested)
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime).resolve()


def list_checkpoints(root: Path) -> list[Path]:
    if not root.is_dir():
        return []
    files: list[Path] = []
    files.extend(sorted(root.glob("checkpoint_epoch_*.pt")))
    latest0 = root / "checkpoint_latest.pt"
    if latest0.is_file():
        files.append(latest0.resolve())
    for run in sorted(root.iterdir()):
        if not run.is_dir():
            continue
        files.extend(sorted(run.glob("checkpoint_epoch_*.pt")))
        latest = run / "checkpoint_latest.pt"
        if latest.is_file():
            files.append(latest.resolve())
    return files


print("Known runs under", OUTPUT_ROOT, "| exists:", OUTPUT_ROOT.is_dir())
for p in sorted([d for d in OUTPUT_ROOT.glob("*") if d.is_dir()], key=lambda x: x.stat().st_mtime, reverse=True)[:8]:
    print(" ", p.name, "| latest:", (p / "checkpoint_latest.pt").is_file())
if (OUTPUT_ROOT / "checkpoint_latest.pt").is_file():
    print(" (also checkpoint_latest.pt directly under OUTPUT_ROOT)")

ckpt_path = resolve_maybe_relative(CHECKPOINT_PATH) if CHECKPOINT_PATH else find_default_checkpoint(OUTPUT_ROOT)
if ckpt_path is None or not ckpt_path.is_file():
    raise FileNotFoundError(
        f"No checkpoint at {ckpt_path!s}. Set CHECKPOINT_PATH (repo-relative ok) or OUTPUT_ROOT. "
        f"Found under OUTPUT_ROOT: {list_checkpoints(OUTPUT_ROOT)[:12]!r}"
    )
print("Using checkpoint:", ckpt_path)

In [ ]:
try:
    raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
except TypeError:
    raw = torch.load(ckpt_path, map_location="cpu")
print("Checkpoint keys:", sorted(raw.keys()))
for k in ("epoch", "global_step"):
    if k in raw:
        print(f"  {k}:", raw[k])

model, experiment_cfg = load_dense_lejepa_from_checkpoint(ckpt_path, map_location=device)
model.eval()
print("Model on device:", next(model.parameters()).device)
print("latent.step_mode (from cfg):", model.config.latent.step_mode)
print("latent hooks:", model.config.latent.resolved_sources())

cfg_json = ckpt_path.parent / "config.json"
if cfg_json.is_file():
    print("config.json present (first keys):", list(json.loads(cfg_json.read_text()).keys())[:6])

In [ ]:
def grayscale_paths_to_batch(paths: list[Path], size: int) -> torch.Tensor:
    from PIL import Image

    tiles: list[torch.Tensor] = []
    for p in paths:
        im = Image.open(p).convert("L")
        try:
            resample = Image.Resampling.BILINEAR
        except AttributeError:
            resample = Image.BILINEAR
        im = im.resize((size, size), resample)
        arr = np.asarray(im, dtype=np.float32) / 255.0
        tiles.append(torch.from_numpy(arr).unsqueeze(0))
    return torch.stack(tiles, dim=0)


in_ch = experiment_cfg.model.in_channels
tiles: list[torch.Tensor] = []

ds = DiscSquareDataset(repeats_per_type=max(1, NUM_IMAGES // 8 + 1), image_size=IMAGE_SIZE)
for i in range(NUM_IMAGES):
    tiles.append(ds[i]["image"])
synthetic = torch.stack(tiles[:NUM_IMAGES], dim=0)

custom = None
if CUSTOM_IMAGE_PATHS:
    custom = grayscale_paths_to_batch([Path(p).resolve() for p in CUSTOM_IMAGE_PATHS], IMAGE_SIZE)

batch_cpu = synthetic.to(dtype=torch.float32)
if custom is not None:
    batch_cpu = torch.cat([batch_cpu, custom], dim=0)

if batch_cpu.shape[1] != in_ch:
    raise ValueError(f"Images have {batch_cpu.shape[1]} channels; model expects in_channels={in_ch}")

print("batch shape:", tuple(batch_cpu.shape))  # B,1,H,W

In [ ]:
# One forward with all hooks (joint). Restore training mode after.
_prev_mode = model.config.latent.step_mode
model.config.latent.step_mode = "joint"
with torch.no_grad():
    outputs = model(batch_cpu.to(device), rotate_latent_index=0)
model.config.latent.step_mode = _prev_mode

lat_by_src = outputs["latents_by_source"]
print("Output keys:", outputs.keys())
print("latents (primary) shape:", tuple(outputs["latents"].shape))
for name, t in lat_by_src.items():
    print(f"  {name}: {tuple(t.shape)}  # B, V, D, H, W")
print("diagnostic losses (not used at inference):", float(outputs["loss"]), float(outputs["inv_loss"]))

In [ ]:
def latent_norm_map(lat_bvdhw: torch.Tensor, b: int, v: int = 0) -> np.ndarray:
    """L2 norm over channel dim -> HxW heatmap."""
    t = lat_bvdhw[b, v].detach().float().cpu()
    m = torch.linalg.vector_norm(t, dim=0).numpy()
    m = m - m.min()
    if m.max() > 0:
        m = m / m.max()
    return m


def show_inputs(batch: torch.Tensor, title: str = "inputs") -> None:
    b = batch.size(0)
    fig, axes = plt.subplots(1, b, figsize=(2.5 * b, 2.5))
    if b == 1:
        axes = [axes]
    for i in range(b):
        axes[i].imshow(batch[i, 0].cpu().numpy(), cmap="gray", vmin=0.0, vmax=1.0)
        axes[i].axis("off")
        axes[i].set_title(f"#{i}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


sources = list(lat_by_src.keys())
n_show = min(batch_cpu.size(0), 6)
fig_h = 2.3 * n_show
fig_w = 2.2 * (1 + len(sources))
fig, axes = plt.subplots(n_show, 1 + len(sources), figsize=(fig_w, fig_h))
if n_show == 1:
    axes = axes[None, :]
for r in range(n_show):
    axes[r, 0].imshow(batch_cpu[r, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[r, 0].axis("off")
    axes[r, 0].set_ylabel(f"img {r}", rotation=90, labelpad=8)
    if r == 0:
        axes[r, 0].set_title("input")
    for j, name in enumerate(sources, start=1):
        axes[r, j].imshow(latent_norm_map(lat_by_src[name], r), cmap="viridis", vmin=0, vmax=1)
        axes[r, j].axis("off")
        if r == 0:
            axes[r, j].set_title(name.replace("_", " "), fontsize=8)
plt.suptitle("Dense latent L2-norm maps (view 0)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def pca_false_color_latent(
    lat_bvdhw: torch.Tensor,
    *,
    b: int = 0,
    view: int = 0,
    percentiles: tuple[float, float] = (2.0, 98.0),
) -> np.ndarray:
    """(B,V,D,H,W) -> (H,W,3) RGB via SVD on D (no sklearn)."""
    t = lat_bvdhw[b, view].detach().float().cpu().numpy()  # (D,H,W)
    d, h, w = t.shape
    x = t.reshape(d, h * w).T
    x = x - x.mean(axis=0, keepdims=True)
    if x.shape[0] < 3 or d < 3:
        raise ValueError("Need at least 3 spatial pixels and 3 latent dims for RGB PCA.")
    u, s, _ = np.linalg.svd(x, full_matrices=False)
    z = u[:, :3] * s[:3]
    rgb = z.reshape(h, w, 3)
    lo, hi = np.percentile(rgb, percentiles)
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    return rgb


# --- pick example ---
PC_B, PC_V, PC_SRC = 0, 0, "encoder_0"  # change PC_SRC to any key in lat_by_src
if PC_SRC not in lat_by_src:
    PC_SRC = next(iter(lat_by_src))
rgb = pca_false_color_latent(lat_by_src[PC_SRC], b=PC_B, view=PC_V)

fig, ax = plt.subplots(1, 2, figsize=(7, 3.2))
ax[0].imshow(batch_cpu[PC_B, 0].numpy(), cmap="gray", vmin=0, vmax=1)
ax[0].set_title("input")
ax[0].axis("off")
ax[1].imshow(rgb)
ax[1].set_title(f"PCA RGB | {PC_SRC} | view {PC_V}")
ax[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def show_latent_all_channels(
    lat_bvdhw: torch.Tensor,
    *,
    b: int = 0,
    view: int = 0,
    source_label: str = "",
    ncols: int = 8,
    cmap: str = "magma",
    share_scale: bool = False,
    percentile_clip: tuple[float, float] | None = (1.0, 99.0),
) -> None:
    """Plot every channel of shape (B,V,D,H,W) as a grid of H×W maps.

    share_scale=True: one vmin/vmax over all D channels (global contrast).
    share_scale=False + percentile_clip: each channel clipped to its own percentiles (easier to see structure).
    """
    t = lat_bvdhw[b, view].detach().float().cpu()
    d, h, w = int(t.shape[0]), int(t.shape[1]), int(t.shape[2])
    nrows = (d + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.15, nrows * 1.15))
    axes_flat = np.atleast_1d(axes).ravel()

    if share_scale:
        arr = t.numpy()
        lo, hi = float(arr.min()), float(arr.max())
        if hi <= lo:
            hi = lo + 1e-8
        vmin, vmax = lo, hi
    else:
        vmin = vmax = None

    for i in range(d):
        ax = axes_flat[i]
        ch = t[i].numpy()
        if not share_scale and percentile_clip is not None:
            lo, hi = np.percentile(ch, percentile_clip)
            if hi <= lo:
                lo, hi = float(ch.min()), float(ch.max()) + 1e-8
            ax.imshow(ch, cmap=cmap, vmin=lo, vmax=hi, aspect="equal")
        else:
            ax.imshow(ch, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_title(str(i), fontsize=8)
        ax.axis("off")

    for j in range(d, len(axes_flat)):
        axes_flat[j].axis("off")

    tag = f"{source_label} | " if source_label else ""
    mode = "global scale" if share_scale else f"per-ch {percentile_clip} clip"
    fig.suptitle(f"{tag}b={b} view={view} | D={d}×{h}×{w} ({mode})", fontsize=11)
    plt.tight_layout()
    plt.show()


# --- all channels for one hook (edit CH_*); repeat for other sources in a loop if you want ---
CH_B, CH_V, CH_SRC = 0, 0, "top"
if CH_SRC not in lat_by_src:
    CH_SRC = next(iter(lat_by_src))
show_latent_all_channels(lat_by_src[CH_SRC], b=CH_B, view=CH_V, source_label=CH_SRC, ncols=8)

# Uncomment to dump every latent *source* (one figure per hook — can be large)
# for _name in lat_by_src:
#     show_latent_all_channels(lat_by_src[_name], b=0, view=0, source_label=_name, ncols=8)

### Linking latents to the image (sanity probes)

**1. Pixel decoder** — Ridge regression from **projected** latent vector at each spatial location (`D` channels) to **input grayscale** on the **latent grid** (targets are downsampled with bilinear resize). Metrics (R², MSE) are on that grid. The figure then **upsamples** the decoded map to **full image size** so you get a **pixel-wise predicted intensity** (density) field aligned with the input.

**2. Edge correlation** — Pearson correlation between **‖z‖₂** per pixel and **Sobel magnitude** of the input (aligned to latent resolution). High values suggest the hook varies where intensity edges vary (weak but cheap).

Increase `NUM_IMAGES` above for more reliable splits.

In [ ]:
import torch.nn.functional as F

# --- settings ---
PROBE_VIEW = 0
RIDGE_LAMBDA = 1e-2
# Train on first N images, test on the rest (needs batch_cpu + lat_by_src from above)
PROBE_TRAIN_N = max(1, batch_cpu.shape[0] - 1)


def _sobel_mag_hw(img_1hw: torch.Tensor) -> torch.Tensor:
    """Grayscale [1,H,W] -> Sobel magnitude [H,W]."""
    x = img_1hw.unsqueeze(0).detach().float()
    kx = torch.tensor(
        [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],
        device=x.device,
        dtype=x.dtype,
    ).view(1, 1, 3, 3)
    ky = torch.tensor(
        [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]],
        device=x.device,
        dtype=x.dtype,
    ).view(1, 1, 3, 3)
    gx = F.conv2d(x, kx, padding=1)
    gy = F.conv2d(x, ky, padding=1)
    return torch.sqrt(gx.squeeze(0).squeeze(0) ** 2 + gy.squeeze(0).squeeze(0) ** 2 + 1e-12)


def _align_img_to_latent_hw(img_b1hw: torch.Tensor, h: int, w: int) -> torch.Tensor:
    return F.interpolate(img_b1hw, size=(h, w), mode="bilinear", align_corners=False).squeeze(1)


def ridge_latent_to_pixel(
    lat_bvdhw: torch.Tensor,
    img_b1hw: torch.Tensor,
    *,
    view: int,
    train_indices: list[int],
    test_indices: list[int],
    ridge: float = 1e-2,
) -> dict:
    """Fit ŷ = w^T z + b with ridge on train images; report test MSE / R²."""
    _, _, d, hl, wl = lat_bvdhw.shape
    y_img = _align_img_to_latent_hw(img_b1hw, hl, wl)

    def stack(idxs: list[int]) -> tuple[torch.Tensor, torch.Tensor]:
        xs, ys = [], []
        for b in idxs:
            z = lat_bvdhw[b, view].reshape(d, -1).T.contiguous()
            targ = y_img[b].reshape(-1)
            xs.append(z)
            ys.append(targ)
        return torch.cat(xs, dim=0), torch.cat(ys, dim=0)

    x_tr, y_tr = stack(train_indices)
    x_te, y_te = stack(test_indices)
    n_tr, _ = x_tr.shape
    x_tr1 = torch.cat([x_tr, torch.ones(n_tr, 1, dtype=x_tr.dtype, device=x_tr.device)], dim=1)
    d1 = d + 1
    xtx = x_tr1.T @ x_tr1
    xtx = xtx + ridge * torch.eye(d1, dtype=x_tr.dtype, device=x_tr.device)
    xty = x_tr1.T @ y_tr
    w = torch.linalg.solve(xtx, xty)
    coef, bias = w[:-1], w[-1]
    y_hat_te = x_te @ coef + bias
    mse = float((y_hat_te - y_te).pow(2).mean())
    y_mean = y_te.mean()
    ss_tot = float((y_te - y_mean).pow(2).sum().clamp_min(1e-12))
    r2 = 1.0 - float((y_te - y_hat_te).pow(2).sum() / ss_tot)
    dumb = float((y_te - y_tr.mean()).pow(2).mean())
    return {
        "mse": mse,
        "r2": r2,
        "mse_predict_train_mean": dumb,
        "n_train_pixels": int(x_tr.shape[0]),
        "n_test_pixels": int(x_te.shape[0]),
    }, (coef, y_hat_te, y_te, test_indices[0])


def latent_l2_vs_edge_corr(
    lat_bvdhw: torch.Tensor,
    img_b1hw: torch.Tensor,
    *,
    b: int,
    view: int,
) -> float:
    _, _, _, hl, wl = lat_bvdhw.shape
    z = lat_bvdhw[b, view]
    zn = torch.linalg.vector_norm(z, dim=0).reshape(-1).float()
    mag = _sobel_mag_hw(img_b1hw[b])
    mag = F.interpolate(mag[None, None], size=(hl, wl), mode="bilinear", align_corners=False)[0, 0]
    m = mag.reshape(-1).float()
    zn = zn - zn.mean()
    m = m - m.mean()
    denom = zn.norm() * m.norm()
    if denom.item() <= 0:
        return float("nan")
    return float((zn * m).sum() / denom)


b_total = batch_cpu.shape[0]
train_ix = list(range(min(PROBE_TRAIN_N, b_total)))
test_ix = list(range(PROBE_TRAIN_N, b_total)) if PROBE_TRAIN_N < b_total else train_ix

if train_ix == test_ix:
    print(
        f"Pixel probe: only {b_total} image(s) — train and test are the SAME indices {train_ix} "
        f"(R² is in-sample; use NUM_IMAGES >= 2 for a hold-out image).\n"
    )
else:
    print(f"Pixel probe: train images {train_ix}, test images {test_ix}\n")

rows = []
last_pred = {}
for name in lat_by_src:
    z = lat_by_src[name]
    met, (coef, y_hat, y_te, _) = ridge_latent_to_pixel(
        z, batch_cpu, view=PROBE_VIEW, train_indices=train_ix, test_indices=test_ix, ridge=RIDGE_LAMBDA
    )
    corrs = [latent_l2_vs_edge_corr(z, batch_cpu, b=j, view=PROBE_VIEW) for j in range(b_total)]
    corr_mean = float(np.nanmean(corrs))
    rows.append((name, met["r2"], met["mse"], met["mse_predict_train_mean"], corr_mean))
    last_pred[name] = (y_hat, y_te, test_ix[0], z.shape[-2:])

# Table
print(f"{'source':<14} {'R²_test':>8} {'MSE_test':>10} {'MSE_dumb':>10} {'‖z‖↔|∇I|':>9}")
print("-" * 60)
for name, r2, mse, dumb, c in rows:
    print(f"{name:<14} {r2:8.4f} {mse:10.6f} {dumb:10.6f} {c:9.4f}")

# Plot: first **test** image — latent-grid decode + **full-res pixel-wise prediction** (upsampled)
PLOT_SRC = "top" if "top" in lat_by_src else next(iter(lat_by_src))
yh, yt, bi, (hh, ww) = last_pred[PLOT_SRC]
flat = hh * ww
np_te = yt[:flat].reshape(hh, ww).numpy()
np_hat_lr = yh[:flat].reshape(hh, ww).detach().numpy()

H0, W0 = int(batch_cpu.shape[-2]), int(batch_cpu.shape[-1])
pred_t = torch.from_numpy(np_hat_lr).float().view(1, 1, hh, ww)
pred_full = F.interpolate(pred_t, size=(H0, W0), mode="bilinear", align_corners=False)[0, 0].numpy()
inp = batch_cpu[bi, 0].numpy()
err = np.abs(inp - pred_full)

fig, ax = plt.subplots(1, 5, figsize=(14, 2.8))
ax[0].imshow(inp, cmap="gray", vmin=0, vmax=1)
ax[0].set_title(f"input #{bi}\n{H0}×{W0}")
ax[0].axis("off")
ax[1].imshow(np_te, cmap="gray", vmin=0, vmax=1)
ax[1].set_title(f"target @{hh}×{ww}\n(downsampled)")
ax[1].axis("off")
ax[2].imshow(np_hat_lr, cmap="gray")
ax[2].set_title(f"ŷ @{hh}×{ww}\n({PLOT_SRC} ridge)")
ax[2].axis("off")
ax[3].imshow(pred_full, cmap="gray", vmin=0, vmax=1)
ax[3].set_title("pixel-wise ŷ\n(bilinear → full res)")
ax[3].axis("off")
vmax_e = float(np.percentile(err, 99.0)) if err.size else 1.0
ax[4].imshow(err, cmap="magma", vmin=0.0, vmax=max(vmax_e, 1e-6))
ax[4].set_title("|input − ŷ|")
ax[4].axis("off")
print(f"Full-res MSE vs input (test #{bi}, {PLOT_SRC}): {float(np.mean((inp - pred_full) ** 2)):.6f}")
fig.suptitle(
    "Latent → linear ridge decode → pixel-wise density prediction (train on other images, eval on this one)",
    fontsize=10,
)
plt.tight_layout()
plt.show()

### Notes

- **Input size:** The spawn demo uses `128×128`. If you change `IMAGE_SIZE`, ensure spatial dimensions still work with the HEA stride pyramid (use multiples of 32 or the size you trained with).
- **All hooks:** We set `model.config.latent.step_mode = "joint"` only for the forward above; your checkpoint on disk is unchanged.
- **Rotate mode:** To mimic training one hook at a time, keep `rotate` and call `model(x, rotate_latent_index=k)` — `latents_by_source` will only contain the active hook for that step.